In [ ]:
%load_ext autoreload
%autoreload 2

## Setup

### Libraries

In [ ]:
import torch
import seaborn as sns
from qgsw.logging import getLogger, setup_root_logger
from qgsw.specs import defaults
import gc

torch.backends.cudnn.deterministic = True
torch.set_grad_enabled(False)
sns.set_theme("notebook",style="dark")

specs = defaults.get()

setup_root_logger(1)
logger = getLogger(__name__)

### Parameters

In [ ]:
from qgsw.configs.core import Configuration
from qgsw.forcing.wind import WindForcing


config = Configuration.from_toml("../output/g5k/param_optim/_config.toml")

H = config.model.h
g_prime = config.model.g_prime
H1, H2 = H[0], H[1]
g1, g2 = g_prime[0], g_prime[1]
beta_plane = config.physics.beta_plane
bottom_drag_coef = config.physics.bottom_drag_coefficient
slip_coef = config.physics.slip_coef

wind = WindForcing.from_config(
    config.windstress,
    config.space,
    config.physics,
)
tx, ty = wind.compute()
p = 4

In [ ]:
H1_,H2_ = 300, 1200 #600, 900 # H[0],H[1]
H_ = torch.tensor([H1_,H2_],**specs)
g1_, g2_ = g_prime[0],g_prime[1]*0.85
g_prime_ = torch.tensor([g1_,g2_],**specs)

### Space

In [ ]:
from qgsw.spatial.core.discretization import SpaceDiscretization3D


space = SpaceDiscretization3D.from_config(
    config.space,
    config.model,
)
dx, dy = space.dx, space.dy

In [ ]:
from qgsw.solver.boundary_conditions.base import Boundaries


def get_psi_slices(imin:int,imax:int,jmin:int,jmax:int) -> tuple[slice]:
    return  [slice(imin, imax + 1), slice(jmin, jmax + 1)]

def extract_psi_w_(psi: torch.Tensor,imin:int,imax:int,jmin:int,jmax:int) -> torch.Tensor:
    """Extract psi."""
    psi_slices_w = get_psi_slices(imin-p,imax+p,jmin-p,jmax+p)
    return psi[..., *psi_slices_w]


def extract_psi_bc(psi: torch.Tensor) -> Boundaries:
    """Extract psi."""
    return Boundaries.extract(psi, p, -p - 1, p, -p - 1, 2)

### Simulation

In [ ]:
dt = 7200
n_steps_per_cyle = 250
separation_steps = 250
comparison_interval = 1
n_cycles = 12

### Outputs

In [ ]:
save_videos = False

### RMSE

In [ ]:
from qgsw.solver.finite_diff import grad_perp, laplacian
from qgsw.utils.reshaping import crop


def rmse(f: torch.Tensor, f_ref: torch.Tensor) -> torch.Tensor:
    """RMSE."""
    return (f - f_ref).square().mean().sqrt() / f_ref.square().mean().sqrt()

def grad_rmse(f:torch.Tensor, f_ref:torch.Tensor) -> torch.Tensor:
    """Gradient RMSE."""
    u,v = grad_perp(f)
    u/=dy
    v/=dx
    u_ref,v_ref = grad_perp(f_ref)
    u_ref/=dy
    v_ref/=dx

    return ((u-u_ref).square().mean()+(v-v_ref).square().mean()).sqrt() / (u_ref.square().mean()+v_ref.square().mean()).sqrt()

def vorticity_rmse(f:torch.Tensor, f_ref:torch.Tensor) -> torch.Tensor:
    """Vorticity RMSE."""
    omega = crop(laplacian(f,dx,dy),1)
    omega_ref = crop(laplacian(f_ref,dx,dy),1)

    return (omega - omega_ref).square().mean().sqrt() / omega_ref.square().mean().sqrt()

## Initial condition

In [ ]:
from qgsw.fields.variables.tuples import UVH
from qgsw.masks import Masks
from qgsw.models.qg.stretching_matrix import compute_A
from qgsw.models.qg.uvh.projectors.core import QGProjector
from qgsw.utils import covphys


P = QGProjector(
    A=compute_A(H=H, g_prime=g_prime),
    H=H.unsqueeze(-1).unsqueeze(-1),
    space=space,
    f0=beta_plane.f0,
    masks=Masks.empty(
        nx=config.space.nx,
        ny=config.space.ny,
    ),
)
uvh0 = UVH.from_file("../output/g5k/param_optim/_data_startup.pt")
psi_start = P.compute_p(covphys.to_cov(uvh0, dx, dy))[0] / beta_plane.f0

### Full-domain model

In [ ]:
from qgsw.models.qg.psiq.core import QGPSIQ


model_3l = QGPSIQ(
    space_2d=space.remove_h(),
    H=H,
    beta_plane=config.physics.beta_plane,
    g_prime=g_prime,
)
model_3l.set_wind_forcing(tx, ty)
model_3l.masks = Masks.empty_tensor(
    model_3l.space.nx,
    model_3l.space.ny,
    device=specs["device"],
)
model_3l.bottom_drag_coef = bottom_drag_coef
model_3l.slip_coef = slip_coef
model_3l.dt = dt
y0 = model_3l.y0

## OBC models

### Base

In [ ]:
from collections.abc import Callable
from pathlib import Path
from typing import TypeVar

from matplotlib import pyplot as plt
import numpy as np

from qgsw.analysis.qg_model import ModelWrapper, ModelsManager
from qgsw.models.qg.psiq.core import QGPSIQCore
from qgsw.spatial.core.discretization import SpaceDiscretization2D

T = TypeVar("T", bound=QGPSIQCore)

class ModelWrapperOBC(ModelWrapper[T]):
    results_paths = Path("../output/g5k/param_optim")

    instantiated=False

    losses:dict[str,list[list[torch.Tensor]]] = {}
    
    ijs:tuple[int,int,int,int] = None
    save_states = False
    show=True

    def __init__(self, space_2d: SpaceDiscretization2D) -> None:
        super().__init__(space_2d)
        self.losses = {
            "rmse": [],"grad_rmse": [], "vorticity_rmse": [],
        }
        self.states:dict[str,list[list[torch.Tensor]]] = {
            "psi1": []
        }

    def _set_params(self) -> None:
        space = self.model.space
        self.model.y0 = y0
        self.model.masks = Masks.empty_tensor(
            space.nx,
            space.ny,
            device=specs["device"],
        )
        self.model.bottom_drag_coef = 0
        self.model.wide = True
        self.model.slip_coef = slip_coef
        self.model.dt = dt
        
    def load(self, imin:int,imax:int,jmin:int,jmax:int)-> dict:
        indices = f"_{imin}_{imax}_{jmin}_{jmax}.pt"
        file = self.results_paths.joinpath(self.prefix+indices)
        return torch.load(file)
    def new_cycle(self) -> None:
        super().new_cycle()
        if self.save_states:
            for s in self.states.values():
                s.append([])

        for loss in self.losses.values():
            loss.append([])
            
    def add_loss(self, loss_value:float,loss_name:str) -> None:
        self.losses[loss_name][-1].append(loss_value)
        
    def plot_loss(self,*,loss_name:str,ax:plt.Axes|None=None,cycle:int|None=None) -> None:
        if not self.show:
            return
        if ax is None:
            ax = plt.gca()
        cycles = [cycle] if cycle is not None else list(range(len(self.losses[loss_name])))
        time_offset= 0
        for i,c in enumerate(cycles):
            times = self.model.dt*np.arange(len(self.losses[loss_name][c]))/3600/24 + time_offset
            time_offset = times[-1] + self.model.dt/3600/24
            loss =  np.array(self.losses[loss_name][c])
            kwargs = self.plot_kwargs.copy()
            if i!= 0:
                kwargs.pop("label")
            ax.plot(times, loss, **kwargs)
        ax.set_xlabel("Time [day]")
    def step(self) -> None:
        super().step()
        if self.save_states:
            self.states["psi1"].append(self.model.psi[:,:1])
            
        
M = TypeVar("M", bound=ModelWrapperOBC[QGPSIQCore])

class ModelsManagerOBC(ModelsManager[M]):

    loss_fn: dict[str, Callable[[torch.Tensor,torch.Tensor], torch.Tensor]]= {
        "rmse":rmse,
        "grad_rmse":grad_rmse,
        "vorticity_rmse":vorticity_rmse
    }

    losses = list(loss_fn.keys())

    def __init__(self, *mw:M) -> None:
        super().__init__(*mw)
        for mw in self.model_wrappers:
            mw.instantiated=True
        self.ijs = self.model_wrappers[0].ijs


    @property
    def ijs(self) -> tuple[int,int,int,int]:
        return self.model_wrappers[0].ijs
    @ijs.setter
    def ijs(self,ijs:tuple[int,int,int,int]) -> None:
        self.loop_over_models(lambda mw: setattr(mw,"ijs",ijs))

    def compute_loss(self, psi_ref:torch.Tensor) -> None:
        for loss_name in self.losses:
            self.loop_over_models(
                lambda mw: mw.add_loss(self.loss_fn[loss_name](mw.model.psi[0,0],psi_ref[0,0]).cpu().item(),loss_name)
            )
        
    def plot_loss(self,*,loss_name:str,ax:plt.Axes|None=None,cycle:int|None=None) -> None:
        self.loop_over_models(lambda mw: mw.plot_loss(loss_name=loss_name,ax=ax,cycle=cycle))

### Reduced gravity

In [ ]:
from torch import Tensor
from qgsw.solver.finite_diff import laplacian
from qgsw.spatial.core.discretization import SpaceDiscretization2D
from qgsw.spatial.core.grid_conversion import interpolate
from qgsw.utils.interpolation import QuadraticInterpolation
from qgsw.utils.reshaping import crop


class ReducedGravity(ModelWrapperOBC[QGPSIQ]):
    prefix = None
    color = "black"
    label="Reduced Gravity"
    H=H[:1]
    g_prime = g_prime[:1]*g_prime[1:2]/(g_prime[:1]+g_prime[1:2])
    def _init_model(self, space_2d:SpaceDiscretization2D) -> None:
        self.model= QGPSIQ(
            space_2d=space_2d,
            H=self.H,
            beta_plane=beta_plane,
            g_prime=self.g_prime,
        )
        self._set_params()
        # self.model.wind_scaling = H[:1].item()
    def compute_q(self,psi: Tensor, beta_effect:torch.Tensor) -> Tensor:
        return interpolate(laplacian(psi,dx,dy) - beta_plane.f0**2 * (1/H1/g1+1/H1/g2)*psi[...,1:-1,1:-1]) + beta_effect
    
    def setup(self, psis: list[torch.Tensor],times:list[torch.Tensor],beta_effect_w:torch.Tensor) -> None:
        psi0 = psis[0]
        psi_bcs = [extract_psi_bc(psi[:,:1]) for psi in psis]
        q_bcs = [
            Boundaries.extract(
                self.compute_q(psi[:, :1],beta_effect_w), 2, -3, 2, -3, 3
            )
            for psi in psis
        ]
        self.model.set_boundary_maps(QuadraticInterpolation(times, psi_bcs), QuadraticInterpolation(times, q_bcs))
        self.model.set_psiq(crop(psi0[:,:1],p), crop(self.compute_q(psi0[:,:1],beta_effect_w),p-1))
        
        if self.save_states:
            self.states["psi1"] = [self.model.psi[:,:1]]
class ReducedGravityPert(ReducedGravity):
    H=H_[:1]
    g_prime_=g_prime_[:1]*g_prime_[1:2]/(g_prime_[:1]+g_prime_[1:2])
    def compute_q(self,psi: Tensor, beta_effect:torch.Tensor) -> Tensor:
        return interpolate(laplacian(psi,dx,dy) - beta_plane.f0**2 * (1/H1_/g1_+1/H1_/g2_)*psi[...,1:-1,1:-1]) + beta_effect

### SurfML

In [ ]:
from torch._tensor import Tensor
from qgsw.decomposition.coefficients import DecompositionCoefs
from qgsw.decomposition.core import build_basis_from_params_dict
from qgsw.models.qg.psiq.modified.forced import QGPSIQRGPsi2TransportDR
from qgsw.models.qg.stretching_matrix import compute_A_tilde
from qgsw.spatial.core.discretization import SpaceDiscretization2D
from qgsw.utils.tensor_dict import change_specs



class SurfML(ModelWrapperOBC[QGPSIQRGPsi2TransportDR]):
    prefix = "results_mixed_rg_ro_ge_nopert"
    color="orange"
    label="GaussBarotropic - RG - NoPert - DR"
    save_video = False
    H = H[:2]
    g_prime=g_prime[:2]
    def __init__(self, space_2d: SpaceDiscretization2D) -> None:
        super().__init__(space_2d)
        self.states["psi2"] = []
        
    def _init_model(self, space_2d:SpaceDiscretization2D) -> None:
        self.model= QGPSIQRGPsi2TransportDR(
            space_2d=space_2d,
            H=self.H,
            beta_plane=beta_plane,
            g_prime=self.g_prime,
        )
        self._set_params()
        self.alphas = {}
        self.coefs = {}
    def compute_q(self,psi: Tensor, A11:torch.Tensor, beta_effect:torch.Tensor) -> Tensor:
        return interpolate(
            laplacian(psi, dx, dy)
            - beta_plane.f0**2 * A11 * psi[..., 1:-1, 1:-1]
        ) + beta_effect
    def setup(self, psis: list[Tensor], times: list[Tensor], beta_effect_w: Tensor) -> None:
        res = self.load(*self.ijs)

        imin,imax,jmin,jmax = self.ijs

        space_slice = space.remove_h().slice(
            imin,imax+1,jmin,jmax+1
        )
        try:
            alpha:torch.Tensor = res[self.cycle]["alpha"]
        except KeyError:
            alpha=torch.tensor(0,**specs)
        self.A = compute_A_tilde(self.H,self.g_prime,alpha,**specs)
        coefs = DecompositionCoefs.from_dict(change_specs(res[self.cycle]["coefs"],**specs))

        self.basis = build_basis_from_params_dict(res[self.cycle]["config"]["basis"])
        try:
            self.basis.freeze_time_normalization(self.model.dt*torch.tensor([n_steps_per_cyle],**specs))
        except:... 
        self.basis.set_coefs(coefs)
        self._fpsi2 = self.basis.localize(
            space_slice.psi.xy.x,space_slice.psi.xy.y
        )

        if self.save_params:
            self.alphas[self.cycle] = alpha
            self.coefs[self.cycle] = coefs
        psi0 = psis[0]
        psi_bcs = [extract_psi_bc(psi[:,:1]) for psi in psis]
        q_bcs = [
                Boundaries.extract(
                    self.compute_q(psi[:, :1],self.A[:1,:1],beta_effect_w), 2, -3, 2, -3, 3
                )
                for psi in psis
            ]

        self.model.set_psiq(crop(psi0[:,:1],p), crop(self.compute_q(psi0[:,:1],self.A[:1,:1],beta_effect_w),p-1))
        self.model.alpha = alpha
        self.model.basis = self.basis
        self.model.set_boundary_maps(QuadraticInterpolation(times, psi_bcs), QuadraticInterpolation(times, q_bcs))
        
        if self.save_states:
            self.states["psi1"] = [self.model.psi[:,:1]]
            self.states["psi2"] = [self._fpsi2(self.model.time)[None,None,...]]

    def step(self) -> None:
        super().step()
        if self.save_states:
            self.states["psi2"].append(self._fpsi2(self.model.time)[None,None,...])
class SurfMLPert(SurfML):
    H=H_[:2]
    g_prime=g_prime_[:2]

### Forced

In [ ]:
from qgsw.decomposition.wavelets import WaveletBasis
from qgsw.models.qg.psiq.modified.forced import QGPSIQForced
from qgsw.spatial.core.discretization import SpaceDiscretization2D

Heq = H[:1]*H[1:2]/(H[:1]+H[1:2])
Heq_ = H_[:1]*H_[1:2]/(H_[:1]+H_[1:2])
class VarDyn(ModelWrapperOBC[QGPSIQForced]):
    prefix = "results_forced_rg_dr"
    color="brown"
    label="Forced DR"
    save_video = False
    H = Heq
    g_prime = g_prime[1:2]
    def __init__(self, space_2d: SpaceDiscretization2D) -> None:
        super().__init__(space_2d)
        self.states["forcing"] = []
    def _init_model(self, space_2d:SpaceDiscretization2D) -> None:
        self.model= QGPSIQForced(
            space_2d=space_2d,
            H=self.H,
            beta_plane=beta_plane,
            g_prime=self.g_prime,
        )
        self.model.wind_scaling = H[:1].item()
        self._set_params()
        self.coefs = {}
    def compute_q(self,psi: Tensor, beta_effect:torch.Tensor) -> Tensor:
        return interpolate(
            laplacian(psi,dx,dy)
            - beta_plane.f0**2 * (1/Heq/g2)*psi[...,1:-1,1:-1]
        ) + beta_effect

    def setup(self, psis: list[Tensor], times: list[Tensor], beta_effect_w: Tensor) -> None:
        res = self.load(*self.ijs)
        coefs = DecompositionCoefs.from_dict(change_specs(res[self.cycle]["coefs"],**specs))
        self.basis = build_basis_from_params_dict(res[self.cycle]["config"]["basis"])
        self.basis.set_coefs(coefs)
        if self.save_params:
            self.coefs[self.cycle] = coefs
            
        self.wv = self.basis.localize(
            self.model.space.remove_h().q.xy.x,
            self.model.space.remove_h().q.xy.y,
        )
        psi0 = psis[0]
        psi_bcs = [extract_psi_bc(psi[:,:1]) for psi in psis]
        q_bcs = [
                Boundaries.extract(
                    self.compute_q(psi[:, :1],beta_effect_w), 2, -3, 2, -3, 3
                )
                for psi in psis
            ]

        self.model.set_psiq(crop(psi0[:,:1],p), crop(self.compute_q(psi0[:,:1],beta_effect_w),p-1))
        self.model.set_boundary_maps(QuadraticInterpolation(times, psi_bcs), QuadraticInterpolation(times, q_bcs))
        
        if self.save_states:
            self.states["psi1"] = [self.model.psi[:,:1]]
            self.states["forcing"] = [crop(self.wv(self.model.time)[None,None,...],p)]
        
    def step(self) -> None:
        self.model.forcing = self.wv(self.model.time)
        super().step()
        if self.save_states:
            self.states["forcing"].append(crop(self.wv(self.model.time)[None,None,...],p))


class VarDynPert(VarDyn):
    H = Heq_
    g_prime = g_prime_[1:2]

    def __init__(self, space_2d: SpaceDiscretization2D) -> None:
        super().__init__(space_2d)
        self.model.wind_scaling = H_[:1].item()

    def compute_q(self,psi: Tensor, beta_effect:torch.Tensor) -> Tensor:
        return interpolate(
            laplacian(psi,dx,dy)
            - beta_plane.f0**2 * (1/Heq_/g2_)*psi[...,1:-1,1:-1]
        ) + beta_effect


In [ ]:
from matplotlib.animation import FuncAnimation
from matplotlib.artist import Artist
from matplotlib.figure import Figure
from matplotlib.gridspec import GridSpec
from matplotlib.image import AxesImage
from matplotlib.lines import Line2D
from mpl_toolkits.axes_grid1 import make_axes_locatable

from qgsw import plots
from qgsw.logging.utils import sec2text
from qgsw.plots.plt_wrapper import retrieve_imshow_data

def build_figure(psis_ref:list[torch.Tensor], times:np.ndarray, *mws:ModelWrapperOBC[T],retrieve_data:Callable[[ModelWrapperOBC],list[torch.Tensor]]=lambda mw, i : mw.psis[i][0,0]) -> Figure:

    fig = plt.figure(figsize=((len(mws)+1)*3,10))
    gs = GridSpec(2,(len(mws)+1),figure=fig, width_ratios=[1]*len(mws)+[1.075])
    ax_ref = fig.add_subplot(gs[0,0])

    axs = [fig.add_subplot(gs[0,i+1]) for i in range(len(mws))]

    ax_l = fig.add_subplot(gs[1,:])
    title = plots.blittable_suptitle(sec2text(0),fig,ax_ref)
    
    vmax = max(crop(psi[0,0],p).abs().max() for psi in psis_ref)

    im_ref = plots.imshow(crop(psis_ref[-1][0,0],p),vmax=vmax,vmin=-vmax,ax=ax_ref,show_cbar=False, title="Reference")
    divider = make_axes_locatable(axs[-1])
    cax = divider.append_axes("right", size="5%", pad=0.05)
    plt.colorbar(im_ref, cax=cax)
    ims :list[AxesImage]= []
    for i,mw in enumerate(mws):
        ims.append(plots.imshow(retrieve_data(mw,0),vmax=vmax,vmin=-vmax,ax=axs[i],show_cbar=False, title = mw.label))
    

    y_max = min(max(np.array(mw.losses["rmse"][-1]).max() for mw in mws),1)
    x_m,y_m = ax_l.margins()

    x_len = (n_steps_per_cyle*dt/24/3600-0)
    y_len = y_max-0

    ax_l.set_xlim(0-x_len*x_m, n_steps_per_cyle*dt/24/3600 + x_len*x_m)
    # ax_l.set_ylim(0-y_len*y_m, y_max+y_len*y_m)
    plots.set_ylims(0,1,ax_l)

    losses:list[Line2D] = []
    dummy_loss = np.ones_like(np.array(mw.losses["rmse"][-1]))*np.nan
    for mw in mws:
        losses += ax_l.plot( times/3600/24, dummy_loss ,**mw.plot_kwargs)

    for ax in axs:
        ax.set_yticks([])

    ax_l.legend(loc="upper left",prop={'size': 8})
    ax_l.set_xlabel("Time [days]")
    ax_l.set_ylabel("Normalized RMSE")
    
    def update(frame:int) -> tuple[Artist,...]:
        title.set_text(sec2text(times[frame]))
        im_ref.set_array(retrieve_imshow_data(crop(psis_ref[frame][0,0],p)))
        for im, mw in zip(ims, mws):
            im.set_array(retrieve_imshow_data(retrieve_data(mw,frame)))
        
        for loss,mw in zip(losses,mws):
            vals = dummy_loss.copy()
            vals[:frame] = np.array(mw.losses["rmse"][-1][:frame])
            loss.set_data(times/3600/24,vals)
        return title, im_ref, *ims, *losses
    return fig, update

def make_psi_video(filename:Path|str, psis_ref:list[torch.Tensor], times:np.ndarray, *mws: ModelWrapperOBC[T])-> None:
    filename = Path(filename)

    if not all(mw.save_states for mw in mws):
        return

    retrieve_data =lambda mw,i : mw.states["psi1"][i][0,0]

    fig, update = build_figure(psis_ref, times, *mws, retrieve_data=retrieve_data)

    FuncAnimation(fig, update, n_steps_per_cyle,blit=True).save(filename,fps=10)
    plt.close()



def build_figure_psi2(psi2s_ref:list[torch.Tensor], times:np.ndarray, *mws:SurfML,retrieve_data:Callable[[ModelWrapperOBC],list[torch.Tensor]]=lambda mw, i : mw.psis[i][0,0]) -> Figure:

    fig = plt.figure(figsize=((len(mws)+1)*3,10))
    gs = GridSpec(2,(len(mws)+1),figure=fig, width_ratios=[1]*len(mws)+[1.075])
    ax_ref = fig.add_subplot(gs[0,0])

    axs = [fig.add_subplot(gs[0,i+1]) for i in range(len(mws))]

    ax_l = fig.add_subplot(gs[1,:])
    title = plots.blittable_suptitle(sec2text(0),fig,ax_ref)
   
    vmax = max(crop(psi[0,0],p).abs().max() for psi in psi2s_ref)

    im_ref = plots.imshow(crop(psi2s_ref[-1][0,0],p),vmax=vmax,vmin=-vmax,ax=ax_ref,show_cbar=False, title="Reference")
    divider = make_axes_locatable(axs[-1])
    cax = divider.append_axes("right", size="5%", pad=0.05)
    plt.colorbar(im_ref, cax=cax)
    ims :list[AxesImage]= []
    for i,mw in enumerate(mws):
        ims.append(plots.imshow(retrieve_data(mw,0),vmax=vmax,vmin=-vmax,ax=axs[i],show_cbar=False, title = mw.label))
    

    y_max = min(max(np.array(mw.psi2_losses[-1]).max() for mw in mws),1)
    x_m,y_m = ax_l.margins()

    x_len = (n_steps_per_cyle*dt/24/3600-0)
    y_len = y_max-0

    ax_l.set_xlim(0-x_len*x_m, n_steps_per_cyle*dt/24/3600 + x_len*x_m)
    plots.set_ylims(0,1,ax_l)

    losses:list[Line2D] = []
    dummy_loss = np.ones_like(np.array(mw.psi2_losses[-1]))*np.nan
    for mw in mws:
        losses += ax_l.plot( times/3600/24, dummy_loss ,**mw.plot_kwargs)

    for ax in axs:
        ax.set_yticks([])

    ax_l.legend(loc="upper left",prop={'size': 8})
    ax_l.set_xlabel("Time [days]")
    ax_l.set_ylabel("Normalized RMSE on Psi2")
    
    def update(frame:int) -> tuple[Artist,...]:
        title.set_text(sec2text(times[frame]))
        im_ref.set_array(retrieve_imshow_data(crop(psi2s_ref[frame][0,0],p)))
        for im, mw in zip(ims, mws):
            im.set_array(retrieve_imshow_data(retrieve_data(mw,frame)))
        
        for loss,mw in zip(losses,mws):
            vals = dummy_loss.copy()
            vals[:frame] = np.array(mw.psi2_losses[-1][:frame])
            loss.set_data(times/3600/24,vals)
        return title, im_ref, *ims, *losses
    return fig, update


def make_psi2_video(filename:Path|str, psi2s_ref:list[torch.Tensor], times:np.ndarray, *mws: SurfML)-> None:
    filename = Path(filename)

    if not all(isinstance(mw,SurfML) for mw in mws):
        return

    retrieve_data =lambda mw,i : mw._fpsi2(torch.tensor([i*dt],**specs))

    fig, update = build_figure_psi2(psi2s_ref, times, *mws, retrieve_data=retrieve_data)

    FuncAnimation(fig, update, n_steps_per_cyle,blit=True).save(filename,fps=10)
    plt.close()

In [ ]:
from qgsw.logging.utils import box, step
from qgsw.solver.finite_diff import grad


def main(imin:int, imax:int,jmin:int,jmax:int, *wrappers:ModelWrapperOBC[QGPSIQ]) -> tuple[ModelsManagerOBC,list[torch.Tensor]]:
    extract_psi_w = lambda psi: extract_psi_w_(psi,imin,imax,jmin,jmax)

    model_3l.reset_time()
    model_3l.set_psi(psi_start)

    space_slice = SpaceDiscretization2D.from_coords(
        x_1d=P.space.remove_h().omega.xy.x[imin : imax + 1, 0],
        y_1d=P.space.remove_h().omega.xy.y[0, jmin : jmax + 1],
    )

    space_slice_w = SpaceDiscretization2D.from_coords(
        x_1d=P.space.remove_h().omega.xy.x[imin - p + 1 : imax + p, 0],
        y_1d=P.space.remove_h().omega.xy.y[0, jmin - p + 1 : jmax + p],
    )
    y_w = space_slice_w.q.xy.y[0, :].unsqueeze(0)
    beta_effect_w = beta_plane.beta * (y_w - y0)

    models = ModelsManagerOBC(
        *wrappers
    )
    models.ijs = (imin,imax,jmin,jmax)
    models.save_params = True

    models.set_wind_forcing(tx[imin:imax, jmin : jmax + 1],ty[imin : imax + 1, jmin:jmax])

    psi0s = {}
    dpsis = {}
    gc.collect()

    for mw in models.model_wrappers:
        mw.save_states=True
        if isinstance(mw,SurfML):
            mw.psi2_losses = []
            mw.dt_psi2_losses = []
            mw.Dt_psi2_losses = []

    for c in range(n_cycles):
        torch.cuda.reset_peak_memory_stats()
        models.new_cycle()
        times = [model_3l.time.item()]

        psi0 = extract_psi_w(model_3l.psi[:,:2])
        psi0s[c] = psi0

        psis = [psi0]

        for _ in range(1, n_steps_per_cyle):
            model_3l.step()

            times.append(model_3l.time.item())

            psi = extract_psi_w(model_3l.psi[:,:2])

            psis.append(psi)
            
        dpsis[c] = (psis[-1]-psis[0])/(n_steps_per_cyle-1)/dt

        models.reset_time()

        models.setup(psis,times,beta_effect_w)

        models.compute_loss(crop(psis[0],p))


        for mw in models.model_wrappers:
            if isinstance(mw,SurfML):
                mw.psi2_losses.append([rmse(mw._fpsi2(mw.model.time), crop(psis[0][0,1],p)).cpu().numpy()])
                mw.dt_psi2_losses.append([])
                mw.Dt_psi2_losses.append([])


        for n in range(1,n_steps_per_cyle):
            models.step()
            models.compute_loss(crop(psis[n],p))
            for mw in models.model_wrappers:
                if "psi2" in mw.states:
                    mw.psi2_losses[-1].append(rmse(mw._fpsi2(mw.model.time), crop(psis[n][0,1],p)).cpu().numpy())
                    mw.dt_psi2_losses[-1].append(rmse(mw._fpsi2.dt(mw.model.time-dt/2), crop(psis[n][0,1]-psis[n-1][0,1],p)/dt).cpu().numpy())


        # make_psi_video(f"../output/videos/area_{imin}_{imax}_{jmin}_{jmax}/psi1_c{c}.mp4",psis,np.array(times)-times[0],rg,forced_g100000000000,forced_noreg,surfml_g100,surfml_noreg,)
        # make_psi2_video(f"../output/videos/area_{imin}_{imax}_{jmin}_{jmax}/psi2_c{c}.mp4",[psi[:1,1:2] for psi in psis],np.array(times)-times[0],surfml_g100,surfml_noreg,surfml_noalpha_g100,surfml_noalpha_noreg,)
        for mw in models.model_wrappers:
                if "psi2" in mw.states:
                    fdxpsi2 = mw.basis.localize_dx(*space_slice.q.xy)
                    fdypsi2 = mw.basis.localize_dy(*space_slice.q.xy)
                    fpsi2 = mw.basis.localize(*space_slice.q.xy)
                    for i in range(249):
                        u,v = grad_perp(mw.states["psi1"][i])
                        u/=dy
                        v/=dx
                        dtpsi2 = (fpsi2(torch.tensor([(i+1)*7200],**specs))-fpsi2(torch.tensor([i*7200],**specs)))/dt
                        dxpsi2 = fdxpsi2(torch.tensor([i*7200],**specs))
                        dypsi2 = fdypsi2(torch.tensor([i*7200],**specs))

                        Dt_ = ((u[...,1:,:]+u[...,:-1,:])/2*dxpsi2+(v[...,1:]+v[...,:-1])/2*dypsi2 + dtpsi2)[0,0]

                        u,v = grad_perp(crop(psis[i][0,0],p))
                        u/=dy
                        v/=dy
                        dtpsi2 = interpolate((crop(psis[i+1][0,1],p)-crop(psis[i][0,1],p))/dt)
                        dxpsi2, dypsi2 = grad(crop(psis[i][0,1],p))
                        dxpsi2/=dx
                        dxpsi2 = (dxpsi2[...,:,1:]+dxpsi2[...,:,:-1])/2
                        dypsi2/=dy
                        dypsi2 = (dypsi2[...,1:,:]+dypsi2[...,:-1,:])/2

                        Dt = ((u[...,1:,:]+u[...,:-1,:])/2*dxpsi2+(v[...,1:]+v[...,:-1])/2*dypsi2 + dtpsi2)
            
                        mw.Dt_psi2_losses[-1].append(rmse(Dt_,Dt).cpu().numpy())

        torch.cuda.empty_cache()
        gc.collect()

        max_mem = torch.cuda.max_memory_allocated() / 1024 / 1024
        msg_mem = f"Cycle {step(c + 1, n_cycles)} | Max memory allocated: {max_mem:.1f} MB."
        logger.info(box(msg_mem, style="round"))

        for _ in range(separation_steps):
            model_3l.step()

    return models, psis

In [ ]:
from matplotlib import pyplot as plt

from qgsw import plots

def show_results(imin:int,imax:int,jmin:int,jmax:int,models:ModelsManagerOBC)-> None:
    show_grad = True
    show_vort = True

    fig,axs = plots.subplots(1+show_grad+show_vort,1,figsize=(21,5+5*show_grad+5*show_vort))
    fig.suptitle(f"QG3L - [{imin}, {imax}] x [{jmin}, {jmax}]")
    plots.set_rowtitles(["RMSE"]+show_grad*[ "Gradient RMSE"] + show_vort*[ "Vorticity RMSE"],axs=axs)
    models.plot_loss(loss_name="rmse",ax=axs[0,0])
    plots.clamp_ylims(0,1,axs[0,0])
    axs[0,0].legend(loc="upper left",prop={'size': 8})
    if show_grad:
        models.plot_loss(loss_name="grad_rmse",ax=axs[1,0])
        plots.clamp_ylims(0,1,axs[1,0])
        axs[1,0].legend(loc="upper left",prop={'size': 8})
    if show_vort:
        models.plot_loss(loss_name="vorticity_rmse",ax=axs[-1,0])
        plots.clamp_ylims(0,1,axs[-1,0])
        axs[-1,0].legend(loc="upper left",prop={'size': 8})

## [32, 96] x [256, 384]

### Model runs

In [ ]:
imin, imax = 32, 96
jmin, jmax = 256, 384


space_slice = SpaceDiscretization2D.from_coords(
    x_1d=P.space.remove_h().omega.xy.x[imin : imax + 1, 0],
    y_1d=P.space.remove_h().omega.xy.y[0, jmin : jmax + 1],
)

rg = ReducedGravity(space_slice)
rg.linestyle="solid"

forced_g1000000000 = VarDyn(space_slice)
forced_g1000000000.label = "Forced - 1000000000"
forced_g1000000000.linestyle = "solid"
forced_g1000000000.prefix = "results_forced_rg_dr_gamma1000000000_obstrack_s250"

surfml_g100 = SurfML(space_slice)
surfml_g100.label = "SurfML -  100"
surfml_g100.color = "navy"
surfml_g100.linestyle = "solid"
surfml_g100.prefix = "results_surfml_gamma100_obstrack_s250"

surfml_noreg = SurfML(space_slice)
surfml_noreg.label = "SurfML -  NR"
surfml_noreg.color = "navy"
surfml_noreg.linestyle = "dashed"
surfml_noreg.prefix = "results_surfml_noreg_obstrack_s250"

psi2 = SurfML(space_slice)
psi2.label = "Psi2"
psi2.color="green"
psi2.linestyle="solid"
psi2.prefix = "results_psi2_o1000_s250"
psi2.results_paths = Path("../output/local/param_optim")

models_z1, psis = main(imin,imax,jmin,jmax, rg, forced_g1000000000, surfml_noreg,surfml_g100,psi2)

### Results

In [ ]:
show_results(imin,imax,jmin,jmax,models_z1)

In [ ]:
forced_g1000000000.label = "VarDyn"
surfml_g100.label = "SurfML"
surfml_noreg.label = "SurfML - No Regularization"

times = np.arange(n_steps_per_cyle)*dt

fig, axs = plots.subplots(1,1, figsize=(10,5),dpi=150)
for mw in [rg, forced_g1000000000, surfml_g100, surfml_noreg]:
    ls = torch.stack([torch.tensor(l,**specs) for l in mw.losses["rmse"]])
    ms = torch.mean(ls,dim=0)
    stds = torch.std(ls,dim=0)
    axs[0,0].fill_between((times-times[0])/3600/24, np.subtract(ms.cpu().numpy(), stds.cpu().numpy()), np.add(ms.cpu().numpy(), stds.cpu().numpy()), alpha=0.1, color=mw.color)
    axs[0,0].plot((times-times[0])/3600/24,ms.cpu().numpy(), **mw.plot_kwargs)
axs[0,0].legend()
axs[0,0].set_xlabel('Time [days]')
axs[0,0].set_ylabel("nRMSE")
axs[0,0].set_title(f"QG3L - [{imin}, {imax}] x [{jmin}, {jmax}]")
plots.clamp_ylims(0,1,axs[0,0])
plots.show()
fig.savefig(f'../output/images/qg3l_rmse_{imin}_{imax}_{jmin}_{jmax}.svg')

fig, axs = plots.subplots(1,1, figsize=(10,5),dpi=150)
for mw in [rg, forced_g1000000000, surfml_g100, surfml_noreg]:
    ls = torch.stack([torch.tensor(l,**specs) for l in mw.losses["grad_rmse"]])
    ms = torch.mean(ls,dim=0)
    stds = torch.std(ls,dim=0)
    axs[0,0].fill_between((times-times[0])/3600/24, np.subtract(ms.cpu().numpy(), stds.cpu().numpy()), np.add(ms.cpu().numpy(), stds.cpu().numpy()), alpha=0.1, color=mw.color)
    axs[0,0].plot((times-times[0])/3600/24,ms.cpu().numpy(), **mw.plot_kwargs)
axs[0,0].legend()
axs[0,0].set_xlabel('Time [days]')
axs[0,0].set_ylabel("Gradient nRMSE")
axs[0,0].set_title(f"QG3L - [{imin}, {imax}] x [{jmin}, {jmax}]")
plots.clamp_ylims(0,1,axs[0,0])
plots.show()
fig.savefig(f'../output/images/qg3l_grad_rmse_{imin}_{imax}_{jmin}_{jmax}.svg')

In [ ]:
from qgsw import plots

fig, axs = plots.subplots(3,1,figsize=(20,15))

plots.set_rowtitles(["RMSE on ψ₂", "RMSE on ψ₂ time derivatives", "RMSE on Dt"],axs)

times_rel = np.arange(n_steps_per_cyle)*dt/3600/24

for mw in [surfml_g100, surfml_noreg]:

    if not mw.instantiated:
        continue

    for c in range(n_cycles):
        
        times = times_rel+(c)*dt*(n_steps_per_cyle)/3600/24

        kwargs = mw.plot_kwargs
        if c!=0:
            kwargs.pop("label")


        axs[0,0].plot(times, np.array(mw.psi2_losses[c]), **kwargs)
      
        axs[1,0].plot((times[1:]+times[:-1])/2, np.array(mw.dt_psi2_losses[c]), **kwargs)
    
        axs[2,0].plot((times[1:]+times[:-1])/2, np.array(mw.Dt_psi2_losses[c]), **kwargs)

plots.set_ylims(0,1,axs[0,0])
axs[0,0].legend()
axs[0,0].set_xlabel("Time [days]")
axs[0,0].set_ylabel("RMSE")
plots.set_ylims(0,1,axs[1,0])
axs[1,0].legend()
axs[1,0].set_xlabel("Time [days]")
axs[1,0].set_ylabel("RMSE")
plots.set_ylims(0,1,axs[2,0])
axs[2,0].legend()
axs[2,0].set_xlabel("Time [days]")
axs[2,0].set_ylabel("RMSE")
plt.show()

In [ ]:
surfml_g100.label = "SurfML"
surfml_noreg.label = "SurfML - No Regularization"

times_rel = np.arange(n_steps_per_cyle)*dt/3600/24

fig, axs = plots.subplots(1,1, figsize=(10,5),dpi=150)
for mw in [surfml_g100, surfml_noreg]:
    ls = torch.stack([torch.tensor([e.item() for e in l],**specs) for l in mw.psi2_losses])
    ms = torch.mean(ls,dim=0)
    stds = torch.std(ls,dim=0)
    axs[0,0].fill_between((times_rel), np.subtract(ms.cpu().numpy(), stds.cpu().numpy()), np.add(ms.cpu().numpy(), stds.cpu().numpy()), alpha=0.1, color=mw.color)
    axs[0,0].plot((times_rel),ms.cpu().numpy(), **mw.plot_kwargs)
axs[0,0].legend()
axs[0,0].set_xlabel('Time [days]')
axs[0,0].set_ylabel("ψ₂ nRMSE")
axs[0,0].set_title(f"QG3L - [{imin}, {imax}] x [{jmin}, {jmax}]")
plots.set_ylims(0,1,axs[0,0])
plots.show()
fig.savefig(f'../output/images/qg3l_rmse_psi2_{imin}_{imax}_{jmin}_{jmax}.svg')

fig, axs = plots.subplots(1,1, figsize=(10,5),dpi=150)
for mw in [surfml_g100, surfml_noreg]:
    ls = torch.stack([torch.tensor([e.item() for e in l],**specs) for l in mw.Dt_psi2_losses])
    ms = torch.mean(ls,dim=0)
    stds = torch.std(ls,dim=0)
    axs[0,0].fill_between(((times_rel)[1:]+(times_rel)[:-1])/2, np.subtract(ms.cpu().numpy(), stds.cpu().numpy()), np.add(ms.cpu().numpy(), stds.cpu().numpy()), alpha=0.1, color=mw.color)
    axs[0,0].plot(((times_rel)[1:]+(times_rel)[:-1])/2,ms.cpu().numpy(), **mw.plot_kwargs)
axs[0,0].legend()
axs[0,0].set_xlabel('Time [days]')
axs[0,0].set_ylabel("Dt ψ₂ nRMSE")
axs[0,0].set_title(f"QG3L - [{imin}, {imax}] x [{jmin}, {jmax}]")
plots.set_ylims(0,1,axs[0,0])
plots.show()
fig.savefig(f'../output/images/qg3l_rmse_Dtpsi2_{imin}_{imax}_{jmin}_{jmax}.svg')

## [112, 176] x [64, 192]

### Model runs

In [ ]:
imin, imax = 112, 176
jmin, jmax = 64, 192


space_slice = SpaceDiscretization2D.from_coords(
    x_1d=P.space.remove_h().omega.xy.x[imin : imax + 1, 0],
    y_1d=P.space.remove_h().omega.xy.y[0, jmin : jmax + 1],
)

rg = ReducedGravity(space_slice)
rg.linestyle="solid"

forced_g1000000000 = VarDyn(space_slice)
forced_g1000000000.label = "Forced - 1000000000"
forced_g1000000000.linestyle = "solid"
forced_g1000000000.prefix = "results_forced_rg_dr_gamma1000000000_obstrack_s250"

surfml_g100 = SurfML(space_slice)
surfml_g100.label = "SurfML -  100"
surfml_g100.color = "navy"
surfml_g100.linestyle = "solid"
surfml_g100.prefix = "results_surfml_gamma100_obstrack_s250"

surfml_noreg = SurfML(space_slice)
surfml_noreg.label = "SurfML -  NR"
surfml_noreg.color = "navy"
surfml_noreg.linestyle = "dashed"
surfml_noreg.prefix = "results_surfml_noreg_obstrack_s250"

psi2 = SurfML(space_slice)
psi2.label = "Psi2"
psi2.color="green"
psi2.linestyle="solid"
psi2.prefix = "results_psi2_o1000_s250"
psi2.results_paths = Path("../output/local/param_optim")

models_z2, psis = main(imin,imax,jmin,jmax, rg, forced_g1000000000, surfml_noreg,surfml_g100,psi2)

### Results

In [ ]:
show_results(imin,imax,jmin,jmax,models_z2)

In [ ]:
forced_g1000000000.label = "VarDyn"
surfml_g100.label = "SurfML"
surfml_noreg.label = "SurfML - No Regularization"

times = np.arange(n_steps_per_cyle)*dt

fig, axs = plots.subplots(1,1, figsize=(10,5),dpi=150)
for mw in [rg, forced_g1000000000, surfml_g100, surfml_noreg]:
    ls = torch.stack([torch.tensor(l,**specs) for l in mw.losses["rmse"]])
    ms = torch.mean(ls,dim=0)
    stds = torch.std(ls,dim=0)
    axs[0,0].fill_between((times-times[0])/3600/24, np.subtract(ms.cpu().numpy(), stds.cpu().numpy()), np.add(ms.cpu().numpy(), stds.cpu().numpy()), alpha=0.1, color=mw.color)
    axs[0,0].plot((times-times[0])/3600/24,ms.cpu().numpy(), **mw.plot_kwargs)
axs[0,0].legend()
axs[0,0].set_xlabel('Time [days]')
axs[0,0].set_ylabel("nRMSE")
axs[0,0].set_title(f"QG3L - [{imin}, {imax}] x [{jmin}, {jmax}]")
plots.clamp_ylims(0,1,axs[0,0])
plots.show()
fig.savefig(f'../output/images/qg3l_rmse_{imin}_{imax}_{jmin}_{jmax}.svg')

fig, axs = plots.subplots(1,1, figsize=(10,5),dpi=150)
for mw in [rg, forced_g1000000000, surfml_g100, surfml_noreg]:
    ls = torch.stack([torch.tensor(l,**specs) for l in mw.losses["grad_rmse"]])
    ms = torch.mean(ls,dim=0)
    stds = torch.std(ls,dim=0)
    axs[0,0].fill_between((times-times[0])/3600/24, np.subtract(ms.cpu().numpy(), stds.cpu().numpy()), np.add(ms.cpu().numpy(), stds.cpu().numpy()), alpha=0.1, color=mw.color)
    axs[0,0].plot((times-times[0])/3600/24,ms.cpu().numpy(), **mw.plot_kwargs)
axs[0,0].legend()
axs[0,0].set_xlabel('Time [days]')
axs[0,0].set_ylabel("Gradietn nRMSE")
axs[0,0].set_title(f"QG3L - [{imin}, {imax}] x [{jmin}, {jmax}]")
plots.clamp_ylims(0,1,axs[0,0])
plots.show()
fig.savefig(f'../output/images/qg3l_grad_rmse_{imin}_{imax}_{jmin}_{jmax}.svg')

In [ ]:
from qgsw import plots

fig, axs = plots.subplots(3,1,figsize=(20,15))

plots.set_rowtitles(["RMSE on ψ₂", "RMSE on ψ₂ time derivatives", "RMSE on Dt"],axs)

times_rel = np.arange(n_steps_per_cyle)*dt/3600/24

for mw in [surfml_g100, surfml_noreg]:

    if not mw.instantiated:
        continue

    for c in range(n_cycles):
        
        times = times_rel+(c)*dt*(n_steps_per_cyle)/3600/24

        kwargs = mw.plot_kwargs
        if c!=0:
            kwargs.pop("label")


        axs[0,0].plot(times, np.array(mw.psi2_losses[c]), **kwargs)
      
        axs[1,0].plot((times[1:]+times[:-1])/2, np.array(mw.dt_psi2_losses[c]), **kwargs)
    
        axs[2,0].plot((times[1:]+times[:-1])/2, np.array(mw.Dt_psi2_losses[c]), **kwargs)

plots.set_ylims(0,1,axs[0,0])
axs[0,0].legend()
axs[0,0].set_xlabel("Time [days]")
axs[0,0].set_ylabel("RMSE")
plots.set_ylims(0,1,axs[1,0])
axs[1,0].legend()
axs[1,0].set_xlabel("Time [days]")
axs[1,0].set_ylabel("RMSE")
plots.set_ylims(0,1,axs[2,0])
axs[2,0].legend()
axs[2,0].set_xlabel("Time [days]")
axs[2,0].set_ylabel("RMSE")
plt.show()

In [ ]:
surfml_g100.label = "SurfML"
surfml_noreg.label = "SurfML - No Regularization"

times_rel = np.arange(n_steps_per_cyle)*dt/3600/24

fig, axs = plots.subplots(1,1, figsize=(10,5),dpi=150)
for mw in [surfml_g100, surfml_noreg]:
    ls = torch.stack([torch.tensor([e.item() for e in l],**specs) for l in mw.psi2_losses])
    ms = torch.mean(ls,dim=0)
    stds = torch.std(ls,dim=0)
    axs[0,0].fill_between((times_rel), np.subtract(ms.cpu().numpy(), stds.cpu().numpy()), np.add(ms.cpu().numpy(), stds.cpu().numpy()), alpha=0.1, color=mw.color)
    axs[0,0].plot((times_rel),ms.cpu().numpy(), **mw.plot_kwargs)
axs[0,0].legend()
axs[0,0].set_xlabel('Time [days]')
axs[0,0].set_ylabel("ψ₂ nRMSE")
axs[0,0].set_title(f"QG3L - [{imin}, {imax}] x [{jmin}, {jmax}]")
plots.set_ylims(0,1,axs[0,0])
plots.show()
fig.savefig(f'../output/images/qg3l_rmse_psi2_{imin}_{imax}_{jmin}_{jmax}.svg')

fig, axs = plots.subplots(1,1, figsize=(10,5),dpi=150)
for mw in [surfml_g100, surfml_noreg]:
    ls = torch.stack([torch.tensor([e.item() for e in l],**specs) for l in mw.Dt_psi2_losses])
    ms = torch.mean(ls,dim=0)
    stds = torch.std(ls,dim=0)
    axs[0,0].fill_between(((times_rel)[1:]+(times_rel)[:-1])/2, np.subtract(ms.cpu().numpy(), stds.cpu().numpy()), np.add(ms.cpu().numpy(), stds.cpu().numpy()), alpha=0.1, color=mw.color)
    axs[0,0].plot(((times_rel)[1:]+(times_rel)[:-1])/2,ms.cpu().numpy(), **mw.plot_kwargs)
axs[0,0].legend()
axs[0,0].set_xlabel('Time [days]')
axs[0,0].set_ylabel("Dt ψ₂ nRMSE")
axs[0,0].set_title(f"QG3L - [{imin}, {imax}] x [{jmin}, {jmax}]")
plots.set_ylims(0,1,axs[0,0])
plots.show()
fig.savefig(f'../output/images/qg3l_rmse_Dtpsi2_{imin}_{imax}_{jmin}_{jmax}.svg')

In [ ]:
from qgsw.observations.satellite_track import SatelliteTrackMask


obs_mask = SatelliteTrackMask(
        space_slice.psi.xy.x,
        space_slice.psi.xy.y,
        track_width=100000,
        track_interval=500000,
        theta=torch.pi / 12,
        full_coverage_time=20 * 3600 * 24,
    )

In [ ]:
n = 240

obs = torch.zeros_like(crop(psis[0][0,0],p))
obs+= obs_mask.at_time(torch.tensor([n*7200],**specs))

In [ ]:
plots.imshow(psi_start[0, 0, 32:97, 256:385])
obs[obs==0]=torch.nan
plots.imshow(obs,show_cbar=False,vmin=0,cmap="Greys",alpha=0.45)

In [ ]:

plots.imshow(psi_start[0, 0, 32:97, 256:385]*obs,)

In [ ]:
fig, axs = plots.subplots(1,3, figsize=(10,5),dpi=150)

axs[0,0].set_ylabel("Y [km]")
axs[0,1].set_yticklabels([])
axs[0,2].set_yticklabels([])

axs[0,0].set_xlabel("X [km]")
axs[0,1].set_xlabel("X [km]")
axs[0,2].set_xlabel("X [km]")

vmax = psi_start[0, 0, 32:97, 256:385].abs().max()

plots.imshow(psi_start[0, 0, 32:97, 256:385],ax=axs[0,0],show_cbar=False,vmin=-vmax,vmax=vmax, title = "Surface stream function field", extent=[0,64*10,0,129*10])
plots.imshow(psi_start[0, 0, 32:97, 256:385],ax=axs[0,1],show_cbar=False,vmin=-vmax,vmax=vmax, title = "Satellite track passage", extent=[0,64*10,0,129*10])
plots.imshow(obs,show_cbar=False,vmin=0,cmap="Greys",alpha=0.75,ax=axs[0,1], extent=[0,64*10,0,129*10])
plots.imshow(psi_start[0, 0, 32:97, 256:385]*obs,ax=axs[0,2],vmin=-vmax,vmax=vmax, title="Available observations",cbar_kwargs={"label":"m².s⁻¹"}, extent=[0,64*10,0,129*10])
plt.tight_layout()
plt.show()
# fig.savefig("../output/images/satellite_track.svg")

In [ ]:
fig, axs = plots.subplots(1,1, figsize=(6,5),dpi=150)
plots.imshow(psi_start[0,0],cbar_kwargs={"label":"m².s⁻¹"}, extent=[0,257*10,0,513*10],ax=axs[0,0])


axs[0,0].set_ylabel("Y [km]")

axs[0,0].set_xlabel("X [km]")

for i,(i1,i2,j1,j2) in enumerate([(32, 96, 64, 192), (32, 96, 256, 384), (112, 176, 64, 192), (112, 176, 256, 384)]):

    axs[0,0].hlines([j1*10,j2*10],i1*10,i2*10,color="black")
    axs[0,0].vlines([i1*10,i2*10],j1*10,j2*10,color="black")

    axs[0,0].text((i1+15)*10, (j1+j2-10)*5,f"Z{i+1}", fontsize=16)

plots.show()
# fig.savefig("../output/images/zones.svg")

# Perturbed

### Z2

In [ ]:
imin, imax = 32, 96
jmin, jmax = 256, 384


space_slice = SpaceDiscretization2D.from_coords(
    x_1d=P.space.remove_h().omega.xy.x[imin : imax + 1, 0],
    y_1d=P.space.remove_h().omega.xy.y[0, jmin : jmax + 1],
)

rg = ReducedGravityPert(space_slice)
rg.linestyle="solid"

forced_g1000000000 = VarDynPert(space_slice)
forced_g1000000000.label = "Forced - 1000000000"
forced_g1000000000.linestyle = "solid"
forced_g1000000000.prefix = "results_forced_rg_dr_perturbed_v2_gamma1000000000_obstrack_s250"

surfml_g100 = SurfMLPert(space_slice)
surfml_g100.label = "SurfML -  100"
surfml_g100.color = "navy"
surfml_g100.linestyle = "solid"
surfml_g100.prefix = "results_surfml_perturbed_v2_gamma100_obstrack_s250"

surfml_noalpha_g100 = SurfMLPert(space_slice)
surfml_noalpha_g100.label = "SurfML -  NR"
surfml_noalpha_g100.color = "lightblue"
surfml_noalpha_g100.linestyle = "solid"
surfml_noalpha_g100.prefix = "results_surfml_perturbed_v2_noalpha_gamma100_obstrack_s250"

psi2 = SurfML(space_slice)
psi2.label = "Psi2"
psi2.color="green"
psi2.linestyle="solid"
psi2.prefix = "results_psi2_o1000_s250"
psi2.results_paths = Path("../output/local/param_optim")

models_z1_pert, psis = main(imin,imax,jmin,jmax, rg, forced_g1000000000, surfml_noalpha_g100,surfml_g100, psi2)

In [ ]:
show_results(imin,imax,jmin,jmax,models_z1_pert)

In [ ]:
forced_g1000000000.label = "VarDyn"
surfml_g100.label = "SurfML"
surfml_noalpha_g100.label = "SurfML - No Alpha"

times = np.arange(n_steps_per_cyle)*dt

fig, axs = plots.subplots(1,1, figsize=(10,5),dpi=150)
for mw in [rg, forced_g1000000000, surfml_g100, surfml_noalpha_g100]:
    ls = torch.stack([torch.tensor(l,**specs) for l in mw.losses["rmse"]])
    ms = torch.mean(ls,dim=0)
    stds = torch.std(ls,dim=0)
    axs[0,0].fill_between((times-times[0])/3600/24, np.subtract(ms.cpu().numpy(), stds.cpu().numpy()), np.add(ms.cpu().numpy(), stds.cpu().numpy()), alpha=0.1, color=mw.color)
    axs[0,0].plot((times-times[0])/3600/24,ms.cpu().numpy(), **mw.plot_kwargs)
axs[0,0].legend()
axs[0,0].set_xlabel('Time [days]')
axs[0,0].set_ylabel("nRMSE")
axs[0,0].set_title(f"QG3L (Perturbed) - [{imin}, {imax}] x [{jmin}, {jmax}]")
plots.clamp_ylims(0,1,axs[0,0])
plots.show()
fig.savefig(f'../output/images/qg3l_pert_rmse_{imin}_{imax}_{jmin}_{jmax}.svg')

fig, axs = plots.subplots(1,1, figsize=(10,5),dpi=150)
for mw in [rg, forced_g1000000000, surfml_g100, surfml_noalpha_g100]:
    ls = torch.stack([torch.tensor(l,**specs) for l in mw.losses["grad_rmse"]])
    ms = torch.mean(ls,dim=0)
    stds = torch.std(ls,dim=0)
    axs[0,0].fill_between((times-times[0])/3600/24, np.subtract(ms.cpu().numpy(), stds.cpu().numpy()), np.add(ms.cpu().numpy(), stds.cpu().numpy()), alpha=0.1, color=mw.color)
    axs[0,0].plot((times-times[0])/3600/24,ms.cpu().numpy(), **mw.plot_kwargs)
axs[0,0].legend()
axs[0,0].set_xlabel('Time [days]')
axs[0,0].set_ylabel("Gradient nRMSE")
axs[0,0].set_title(f"QG3L (Perturbed) - [{imin}, {imax}] x [{jmin}, {jmax}]")
plots.clamp_ylims(0,1,axs[0,0])
plots.show()
fig.savefig(f'../output/images/qg3l_pert_grad_rmse_{imin}_{imax}_{jmin}_{jmax}.svg')

### Z3

In [ ]:
imin, imax = 112, 176
jmin, jmax = 64, 192


space_slice = SpaceDiscretization2D.from_coords(
    x_1d=P.space.remove_h().omega.xy.x[imin : imax + 1, 0],
    y_1d=P.space.remove_h().omega.xy.y[0, jmin : jmax + 1],
)

rg = ReducedGravityPert(space_slice)
rg.linestyle="solid"

forced_g1000000000 = VarDynPert(space_slice)
forced_g1000000000.label = "Forced - 1000000000"
forced_g1000000000.linestyle = "solid"
forced_g1000000000.prefix = "results_forced_rg_dr_perturbed_v2_gamma1000000000_obstrack_s250"

surfml_g100 = SurfMLPert(space_slice)
surfml_g100.label = "SurfML -  100"
surfml_g100.color = "navy"
surfml_g100.linestyle = "solid"
surfml_g100.prefix = "results_surfml_perturbed_v2_gamma100_obstrack_s250"

surfml_noalpha_g100 = SurfMLPert(space_slice)
surfml_noalpha_g100.label = "SurfML -  NR"
surfml_noalpha_g100.color = "lightblue"
surfml_noalpha_g100.linestyle = "solid"
surfml_noalpha_g100.prefix = "results_surfml_perturbed_v2_noalpha_gamma100_obstrack_s250"

psi2 = SurfML(space_slice)
psi2.label = "Psi2"
psi2.color="green"
psi2.linestyle="solid"
psi2.prefix = "results_psi2_o1000_s250"
psi2.results_paths = Path("../output/local/param_optim")

models_z2_pert, psis = main(imin,imax,jmin,jmax, rg, forced_g1000000000, surfml_noalpha_g100,surfml_g100,psi2)

In [ ]:
show_results(imin,imax,jmin,jmax,models_z2_pert)

In [ ]:
forced_g1000000000.label = "VarDyn"
surfml_g100.label = "SurfML"
surfml_noalpha_g100.label = "SurfML - No Alpha"

times = np.arange(n_steps_per_cyle)*dt

fig, axs = plots.subplots(1,1, figsize=(10,5),dpi=150)
for mw in [rg, forced_g1000000000, surfml_g100, surfml_noalpha_g100]:
    ls = torch.stack([torch.tensor(l,**specs) for l in mw.losses["rmse"]])
    ms = torch.mean(ls,dim=0)
    stds = torch.std(ls,dim=0)
    axs[0,0].fill_between((times-times[0])/3600/24, np.subtract(ms.cpu().numpy(), stds.cpu().numpy()), np.add(ms.cpu().numpy(), stds.cpu().numpy()), alpha=0.1, color=mw.color)
    axs[0,0].plot((times-times[0])/3600/24,ms.cpu().numpy(), **mw.plot_kwargs)
axs[0,0].legend()
axs[0,0].set_xlabel('Time [days]')
axs[0,0].set_ylabel("nRMSE")
axs[0,0].set_title(f"QG3L (Perturbed) - [{imin}, {imax}] x [{jmin}, {jmax}]")
plots.clamp_ylims(0,1,axs[0,0])
plots.show()
fig.savefig(f'../output/images/qg3l_pert_rmse_{imin}_{imax}_{jmin}_{jmax}.svg')

fig, axs = plots.subplots(1,1, figsize=(10,5),dpi=150)
for mw in [rg, forced_g1000000000, surfml_g100, surfml_noalpha_g100]:
    ls = torch.stack([torch.tensor(l,**specs) for l in mw.losses["grad_rmse"]])
    ms = torch.mean(ls,dim=0)
    stds = torch.std(ls,dim=0)
    axs[0,0].fill_between((times-times[0])/3600/24, np.subtract(ms.cpu().numpy(), stds.cpu().numpy()), np.add(ms.cpu().numpy(), stds.cpu().numpy()), alpha=0.1, color=mw.color)
    axs[0,0].plot((times-times[0])/3600/24,ms.cpu().numpy(), **mw.plot_kwargs)
axs[0,0].legend()
axs[0,0].set_xlabel('Time [days]')
axs[0,0].set_ylabel("Gradient nRMSE")
axs[0,0].set_title(f"QG3L (Perturbed) - [{imin}, {imax}] x [{jmin}, {jmax}]")
plots.clamp_ylims(0,1,axs[0,0])
plots.show()
fig.savefig(f'../output/images/qg3l_pert_grad_rmse_{imin}_{imax}_{jmin}_{jmax}.svg')